# GLiNER multi-v2.1 — DIMER zero-shot named-entity recognition tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/gliner-ner-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/gliner-ner-pipeline/blob/main/tutorials/gliner_ner_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-urchade%2Fgliner__multi--v2.1-ffcc4d?style=flat)](https://huggingface.co/urchade/gliner_multi-v2.1)
[![Upstream](https://img.shields.io/badge/Upstream-urchade%2FGLiNER-181717?style=flat&logo=github&logoColor=white)](https://github.com/urchade/GLiNER)
[![arXiv](https://img.shields.io/badge/arXiv-2311.08526-b31b1b.svg)](https://arxiv.org/abs/2311.08526)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.0  
**Capability:** zero-shot named-entity recognition with a caller-supplied label set using the pinned GLiNER multi-v2.1 weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API (`GLiNERPipeline`) rather than reimplementing model inference. At inference the text and the caller's entity-type names are encoded together by the mDeBERTa-v3 bidirectional encoder inside GLiNER, every candidate span is scored against every label, and spans whose **score** (a per-span confidence in [0, 1], **not a calibrated probability**) reaches the `threshold` are returned with character offsets. The label set is free text chosen by the caller at call time — that is what "zero-shot" means here — and the threshold (default `DEFAULT_THRESHOLD` = 0.5, the upstream default) is exposed and **owned by the caller**. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting — the pinned checkpoint is used as published. **Two pinned snapshots** are involved: the GLiNER weights and the mDeBERTa tokenizer/config the encoder needs; the repository verifies both against their manifests before loading. What upstream supplies is the model, the encoder assets and the span-decoding library; what this repository adds is dual-manifest verification, input validation and ceilings, offset-checked output, the `entity_f1` helper, and a fixed output contract.

**Learning objectives:** bootstrap the repository in a fresh runtime, author a synthetic input text and label set (or upload your own), surface the pipeline's ceilings and the threshold semantics, stage and digest-verify **both** immutable snapshots, detect entities through the public API, read spans and scores correctly, understand when `entity_f1` applies and why no metric is reported without gold annotations, and export the entities with identifiers plus provenance.

**This notebook does not demonstrate:** entity linking or normalisation, relation extraction, coreference, document-level processing beyond `MAX_TEXT_CHARS` (the caller chunks), fine-tuning, or any calibrated confidence. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available; the model card's CPU smoke verified both snapshots, loaded, and detected five spans in one sentence in 14.6 s wall clock, so the default sentence runs in well under a minute on a hosted CPU runtime. The pinned `torch==2.14.0` install and the 1.16 GB GLiNER checkpoint are the largest downloads of the run.
- **Knowledge:** basic Python; what character offsets are; what precision/recall/F1 over spans mean.
- **Data:** the default sample is a synthetic sentence and label set authored in code; BYOD is one UTF-8 text file plus a label list, gated off by default. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded text remains in the notebook runtime; this pipeline does not send it to a third-party inference API.
- **External access:** the Git clone, the pinned wheel installs, and the fetch of the missing GLiNER weight file from the Hugging Face Hub at the immutable revision (the encoder tokenizer/config files are committed). No credentials are needed.
- **Expected warning:** while building the fast DeBERTa tokenizer from `spm.model`, `transformers==4.57.6` logs an "incorrect regex pattern … `fix_mistral_regex`" warning. It refers to a Mistral tokenizer issue, does not apply to this SentencePiece model, and is documented in the README and model card; the pipeline captures it in `load_warnings` and Section 4 prints it so you can see it is the expected one.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `gliner`, `transformers`, `huggingface-hub`, `safetensors`, `numpy`, `protobuf`, `sentencepiece`) are directly pinned by `pyproject.toml`. If installation replaces any package that this runtime has already imported (hosted runtimes commonly pre-import a different NumPy or Pillow), the cell fails with a restart instruction rather than continuing with mixed versions: restart the runtime and rerun from the top. Inference runs in float32 on CPU and on CUDA when available; no half precision, compilation, or quantization is applied.

In [ ]:
import importlib
import importlib.metadata
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/gliner-ner-pipeline.git'
REPO_NAME = 'gliner-ner-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, gliner, torch, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'gliner': gliner.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Author the synthetic sample or optional BYOD

The default sample is **synthetic**: one English sentence and a four-type label set written in this cell, so it needs no download and names no real person (the names are invented). It ships **no gold annotations** — the sentence was written to contain obvious entities, but the notebook does not assert where they are — so the spans it produces are smoke/sanity evidence that the code path works, never an accuracy measurement and never benchmark evidence. If you want a metric, paste gold annotations into `GOLD_JSON` (a JSON list of `{"start", "end", "label"}` objects with character offsets into the text, matching the label names exactly); Section 5 then scores exact-span micro precision/recall/F1 with the repository's `entity_f1`. Leave it empty to report no metric.

BYOD is optional and disabled by default. Expected BYOD input: one UTF-8 text file of at most `MAX_TEXT_CHARS` characters (the library also cuts text beyond 384 words — chunk longer documents yourself), plus the comma-separated `LABELS` form field naming 1 to `MAX_LABELS` unique entity types (each at most 100 characters). The upload stays inside this runtime.

In [ ]:
import hashlib
import io
import json

USE_BYOD = False  # @param {type:"boolean"}
LABELS = 'person, organization, location, date'  # @param {type:"string"}
THRESHOLD = 0.5  # @param {type:"number"}
GOLD_JSON = ''  # @param {type:"string"}
labels = [label.strip() for label in LABELS.split(',') if label.strip()]
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    sample_name = next(iter(uploaded))
    text = uploaded[sample_name].decode('utf-8').strip()
    sample_kind = 'BYOD upload'
else:
    text = 'Elena Marquez joined the Lakeside Research Institute in Wellington on 3 March 2021 after leaving Orion Analytics.'
    sample_name = 'synthetic_sentence'
    sample_kind = 'synthetic (authored in this cell; invented names)'
gold = json.loads(GOLD_JSON) if GOLD_JSON.strip() else None
if gold is not None:
    for index, item in enumerate(gold):
        if not (isinstance(item, dict) and {'start', 'end', 'label'} <= set(item)):
            raise ValueError(f'GOLD_JSON[{index}] must be an object with start, end and label')
        if not (0 <= int(item['start']) < int(item['end']) <= len(text)) or item['label'] not in labels:
            raise ValueError(f'GOLD_JSON[{index}] has offsets outside the text or a label not in LABELS: {item}')
sample_sha256 = hashlib.sha256(text.encode('utf-8')).hexdigest()
print({'sample': sample_name, 'sample_kind': sample_kind, 'chars': len(text), 'words': len(text.split()), 'labels': labels, 'threshold': THRESHOLD, 'has_gold': gold is not None, 'text_sha256': sample_sha256})
print(text)

## 3. Validate the input against the pipeline ceilings

The pipeline enforces two operational ceilings, imported here from the package so the values shown are the ones in force: `MAX_TEXT_CHARS` (characters per `detect` call; the caller chunks longer text) and `MAX_LABELS` (entity types per call, the most the checkpoint saw per example in training); labels must be unique, non-empty and at most 100 characters, and `threshold` must lie in [0, 1]. This cell surfaces the ceilings and checks the input before any model work, naming the failing condition and the corrective action; `detect()` re-applies the same rules authoritatively and raises `TypeError`/`ValueError` on its own. **What can change your text:** the GLiNER library cuts input beyond 384 words (`max_len` in the snapshot config) without reporting where — the word count is printed so you can see whether the sample is near that limit; the notebook itself does not alter the text.

In [ ]:
from gliner_ner_pipeline import DEFAULT_THRESHOLD, MAX_LABELS, MAX_TEXT_CHARS

ceilings = {'MAX_TEXT_CHARS': MAX_TEXT_CHARS, 'MAX_LABELS': MAX_LABELS, 'library_word_limit': 384, 'DEFAULT_THRESHOLD': DEFAULT_THRESHOLD}
print(ceilings)
problems = []
if not text.strip():
    problems.append('text is empty: supply a non-empty file')
if len(text) > MAX_TEXT_CHARS:
    problems.append(f'text has {len(text)} chars > MAX_TEXT_CHARS={MAX_TEXT_CHARS}: chunk the document and run detect per chunk')
if not 1 <= len(labels) <= MAX_LABELS:
    problems.append(f'{len(labels)} labels is outside 1..MAX_LABELS={MAX_LABELS}: edit the LABELS form field')
if len(set(labels)) != len(labels):
    problems.append('labels must be unique: remove duplicates from LABELS')
if any(len(label) > 100 for label in labels):
    problems.append('a label exceeds 100 characters: shorten it')
if not 0.0 <= THRESHOLD <= 1.0:
    problems.append(f'THRESHOLD {THRESHOLD} is outside [0, 1]: fix the form value')
if problems:
    raise ValueError('input rejected before model execution: ' + '; '.join(problems))
print({'chars': len(text), 'words': len(text.split()), 'near_word_limit': len(text.split()) > 300, 'labels': len(labels), 'within_ceilings': True})

## 4. Stage, verify, and resolve both pinned snapshots

The public API pins two immutable identities, both imported from the package and never typed here: the GLiNER checkpoint (`MODEL_ID` at `MODEL_REVISION`) and the mDeBERTa-v3-base encoder assets (`ENCODER_MODEL_ID` at `ENCODER_REVISION`) whose tokenizer and config GLiNER needs. The repository commits both DIMER manifests (`weights/gliner-multi-v2.1/dimer-base-manifest.json`, 3 entries; `weights/mdeberta-v3-base-tokenizer/dimer-base-manifest.json`, 4 entries) and the small config/tokenizer files, but git-ignores the 1.16 GB `model.safetensors`, so a fresh clone must stage that file first. `stage_missing_files(WEIGHTS_DIR, allow_download=True)` and `stage_missing_encoder_files(ENCODER_DIR, allow_download=True)` each fetch only the manifest-listed files absent from their directory, at their pinned revision, and return what they fetched (`['model.safetensors']` and `[]` on a fresh clone); each refuses to stage if its manifest disagrees with the package's pinned identity. `verify_snapshot()` and `verify_encoder_snapshot()` then re-hash every listed file and raise on the first size or digest mismatch; only afterwards does `from_pretrained` load through the `gliner` library with the encoder redirected to the verified directory and `local_files_only=True` — no Hub access and no unpinned encoder download. Both identities, both verification summaries, the selected device and the captured load warnings are printed before inference; expect exactly the `fix_mistral_regex` warning described in the prerequisites.

In [ ]:
from gliner_ner_pipeline import ENCODER_KEY, ENCODER_MODEL_ID, ENCODER_REVISION, MODEL_ID, MODEL_KEY, MODEL_REVISION, GLiNERPipeline, entity_f1, stage_missing_encoder_files, stage_missing_files, verify_encoder_snapshot, verify_snapshot

print({'model_id': MODEL_ID, 'revision': MODEL_REVISION})
print({'encoder_model_id': ENCODER_MODEL_ID, 'encoder_revision': ENCODER_REVISION})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
ENCODER_DIR = ROOT / 'weights' / ENCODER_KEY
# Only manifest-listed files that are absent are fetched, at the immutable revisions the package
# pins; the two verify calls then check every byte count and SHA-256 before anything is loaded.
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
fetched_encoder = stage_missing_encoder_files(ENCODER_DIR, allow_download=True)
print({'fetched': fetched, 'fetched_encoder': fetched_encoder})
snapshot_info = verify_snapshot(WEIGHTS_DIR)
encoder_info = verify_encoder_snapshot(ENCODER_DIR)
print({'snapshot': {'path': str(WEIGHTS_DIR), 'model_id': snapshot_info['modelId'], 'revision': snapshot_info['revision'], 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')}})
print({'encoder_snapshot': {'path': str(ENCODER_DIR), 'model_id': encoder_info['modelId'], 'revision': encoder_info['revision'], 'files': len(encoder_info['files']), 'total_bytes': encoder_info.get('totalBytes')}})
pipe = GLiNERPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, encoder_dir=ENCODER_DIR)
print({'device': pipe.device, 'precision': 'float32', 'load_warnings': pipe.load_warnings})

## 5. Detect entities and evaluate when gold annotations exist

`detect(text, labels, threshold=...)` returns `entities` — a list of `{text, label, start, end, score}` with character offsets into the input (the pipeline checks every span lies inside the text and carries one of your labels) — plus `n_entities`, the `labels` and `threshold` used, and both model identities. **Score semantics:** each `score` is the library's per-span confidence in [0, 1] for that span-label pair; it is not a calibrated probability, and the only decision the pipeline makes is the cutoff `score >= threshold`. The default 0.5 is the upstream default, not a validated operating point: lowering it returns more (and less certain) spans, raising it fewer; the caller owns choosing it on their own annotated data.

**Evaluation:** the repository ships `entity_f1(predicted, gold)` — exact-span micro precision, recall and F1, where a hit is an identical `(start, end, label)` triple — the standard NER measure, which credits nothing for a span that is off by one character or carries a different label. It applies only when gold annotations exist. The synthetic sample has none, so **no metric is reported** on the default path and the cell prints that fact instead of a number; when you supply `GOLD_JSON`, the figure is a single-text tutorial metric with no dispersion estimate, not a benchmark result, and a real evaluation needs an annotated corpus in your domain and language with your label definitions. No baseline is reported: a trivial baseline (no entities) has F1 = 0 by construction and teaches nothing without gold. The runtime figure is measured on the runtime identified in Section 1 for this one text and includes the first-call warm-up.

In [ ]:
import time

started = time.perf_counter()
result = pipe.detect(text, labels, threshold=THRESHOLD)
elapsed = time.perf_counter() - started
entities = result['entities']
checks = {
    'offsets_inside_text': all(0 <= e['start'] < e['end'] <= len(text) for e in entities),
    'span_text_matches_offsets': all(text[e['start']:e['end']] == e['text'] for e in entities),
    'labels_from_request': all(e['label'] in labels for e in entities),
    'scores_at_or_above_threshold': all(THRESHOLD <= e['score'] <= 1.0 for e in entities),
}
if not all(checks.values()):
    raise RuntimeError(f'detect output failed a sanity check: {checks}')
print({key: value for key, value in result.items() if key != 'entities'})
print({'seconds': round(elapsed, 3), 'checks': checks})
for index, entity in enumerate(entities):
    print(f"{index:>2}. [{entity['start']:>3}:{entity['end']:<3}] {entity['label']:<14} score {entity['score']:.3f}  {entity['text']}")
metrics = {}
if gold is not None:
    metrics['entity_f1'] = entity_f1(entities, gold)
    print({'sample_metrics': metrics, 'estimation': 'single text, exact-span micro scores; tutorial evidence only'})
else:
    print('no metric is reported: the sample has no gold annotations, so entity_f1 is not computed; the spans above are sanity evidence only')

## 6. Export entities and provenance

One JSON record is written under `outputs/`: the input text and its digest, the label set and threshold, an `entities` list with an index per span plus its text, label, character offsets and score (so every span maps back to its input), the metric block (empty without gold), the sanity checks, the ceilings in force, the repository revision, **both** model identifiers and immutable revisions, both verified snapshot summaries, the captured load warnings, and the runtime identity (Python, `torch`, `gliner`, `transformers`, device, precision). No credentials are involved in any step, so none can reach the export.

In [ ]:
os.makedirs('outputs', exist_ok=True)
payload = {
    'text': text,
    'labels': result['labels'],
    'threshold': result['threshold'],
    'entities': [{'index': index, **entity} for index, entity in enumerate(entities)],
    'n_entities': result['n_entities'],
    'metrics': metrics,
    'gold': gold,
    'sanity_checks': checks,
    'ceilings': ceilings,
    'sample': {'name': sample_name, 'kind': sample_kind, 'text_sha256': sample_sha256},
    'seconds': round(elapsed, 3),
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'encoder_model_id': ENCODER_MODEL_ID,
    'encoder_revision': ENCODER_REVISION,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot_info['files']), 'total_bytes': snapshot_info.get('totalBytes')},
    'encoder_snapshot': {'path': str(ENCODER_DIR), 'files': len(encoder_info['files']), 'total_bytes': encoder_info.get('totalBytes')},
    'load_warnings': pipe.load_warnings,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'gliner': gliner.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
        'precision': 'float32',
    },
}
with open('outputs/gliner_ner_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))
print('outputs/gliner_ner_result.json')

## Interpretation and limits

Each returned span is a model prediction for a label name you chose: the `score` is a per-span confidence in [0, 1], not a calibrated probability, and the only decision rule is the caller-owned `threshold` (default 0.5, the upstream default, not a validated operating point). On the synthetic sentence the spans are plumbing evidence only; no metric is reported without gold annotations, and an `entity_f1` figure on one text has no dispersion and generalises to nothing. The label names are part of the input — different wordings of the same concept give different spans — and the checkpoint's multilingual quality varies by language and domain in ways this notebook does not measure. Text beyond 384 words is cut silently by the library (the notebook prints the word count; chunk long documents), nested or overlapping entities may be lost, and entity linking, normalisation, relations and coreference are not provided. Two supply chains are involved — the GLiNER weights and the mDeBERTa encoder assets — and both were digest-verified before loading. Inference is deterministic given the same weights, device and library versions; CPU and CUDA scores can differ slightly and move borderline spans across the threshold.

Successful execution proves that the recorded repository revision can bootstrap in a fresh runtime, stage and digest-verify both pinned snapshots, validate the demonstrated input against the enforced ceilings, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, extraction quality on any domain or language, a validated threshold, safety for high-consequence decisions, or production fitness on an unseen domain.

**Troubleshooting.** `RuntimeError: Core dependencies changed while older modules were loaded` in Section 1: the pinned install replaced a package the runtime had pre-imported — restart the runtime and rerun from the top. `FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 4: a staged file is incomplete or altered — delete it from `weights/gliner-multi-v2.1/` (or `weights/mdeberta-v3-base-tokenizer/`) and rerun Section 4. A `fix_mistral_regex` warning in Section 4: expected and harmless (see prerequisites). A `ValueError` naming `MAX_TEXT_CHARS`, `MAX_LABELS` or `THRESHOLD` in Section 3: fix the form values or chunk the text and rerun from Section 2. Zero entities returned: lower `THRESHOLD`, or check that your label names describe the entities in plain words.

**Next experiments.** Paste gold annotations for the default sentence into `GOLD_JSON` to see how exact-span F1 penalises a boundary that is off by one character; sweep `THRESHOLD` over 0.3-0.7 on a text you can judge and watch precision and recall trade off; reword a label (for example `city` versus `location`) and observe how the spans change. None of these turns the sample result into evidence of production fitness.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance (both snapshots): `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/urchade/gliner_multi-v2.1
- Encoder assets: https://huggingface.co/microsoft/mdeberta-v3-base
- Upstream code: https://github.com/urchade/GLiNER
- GLiNER paper: https://arxiv.org/abs/2311.08526